# Rota Segura — Modelos de risco de alagamento (Data Science)

**Pergunta:** dada a chuva medida no dia, qual a chance de a região alagar (regressão logística) e quantos pontos de alagamento esperar (regressão linear)?

O resultado alimenta o **alerta antecipado** do serviço Python (`/risk`). O alerta por nível d'água em cm continua no Java e não depende deste modelo.

**Unidade de análise:** um dia (horário de Brasília).

**Fontes**
- Chuva: DADOS DA REDE OBSERVACIONAL DO CEMADEN/MCTIC, Plataforma de Entrega de Dados (https://ped.cemaden.gov.br), estação Itaim Paulista (355030881A), 2016–2025, sensor 10 (mm por leitura). Horário em UTC, convertido para America/Sao_Paulo. Unidade assumida em mm (não informada no arquivo).
- Alagamentos (rótulo): PREFEITURA DE SÃO PAULO. Centro de Gerenciamento de Emergências Climáticas (CGE). Histórico de pontos de alagamento, https://www.cgesp.org/v3/alagamentos.jsp, 2016–2025, acesso em nov. 2026, raspagem automatizada (1 req./4 s).
- Validação no Alto Tietê: Cemaden, Reindesc (registro de ocorrências de desastres), 2016–2026.

**Integrantes:** ver `integrantes.txt`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score,
                             f1_score, average_precision_score, precision_recall_curve,
                             mean_absolute_error, mean_squared_error, r2_score)
import joblib

RANDOM_STATE = 42
DADOS = Path("dados")
ARQ_CHUVA = DADOS / "ped" / "PED_Cemaden_355030881A.csv"   # Itaim Paulista
ARQ_ROTULO = DADOS / "cge_diario.csv"
REGIAO = "extremo_leste"          # ou "zona_leste" (análise de sensibilidade)
MIN_HORAS_COM_DADO = 20           # dia com dado = pelo menos 20 das 24 horas com leitura
RECALL_MINIMO = 0.70              # regra do limite de "alta probabilidade" (decisions.md)
TREINO_ATE, VALIDACAO = "2022-12-31", ("2023-01-01", "2023-12-31")   # teste: 2024–2025

## 1. Chuva: leituras do Cemaden → variáveis diárias

In [ ]:
bruto = pd.read_csv(ARQ_CHUVA)
chuva = (bruto[bruto["id_sensor"] == 10]
         .drop_duplicates("datahora")
         .assign(hora=lambda d: pd.to_datetime(d["datahora"]).dt.tz_localize("UTC").dt.tz_convert("America/Sao_Paulo"))
         .set_index("hora")["valor"].sort_index())

# Por hora: soma das leituras; hora sem nenhuma leitura fica NaN (falha da estação, não chuva zero)
por_hora = chuva.resample("1h").sum(min_count=1)
max_3h = por_hora.rolling(3, min_periods=3).sum()

dia = pd.DataFrame({
    "chuva_24h": por_hora.resample("1D").sum(min_count=1),
    "chuva_max_1h": por_hora.resample("1D").max(),
    "chuva_max_3h": max_3h.resample("1D").max(),
    "horas_com_leitura": por_hora.notna().resample("1D").sum(),
})
dia.index = dia.index.tz_localize(None).normalize()
dia["dia_com_dado"] = dia["horas_com_leitura"] >= MIN_HORAS_COM_DADO

# Variáveis que só olham para trás (dias anteriores)
dia["chuva_dia_anterior"] = dia["chuva_24h"].shift(1)
dia["chuva_72h"] = dia["chuva_24h"].rolling(3, min_periods=3).sum()
dia["chuva_7d_anteriores"] = dia["chuva_24h"].shift(1).rolling(7, min_periods=5).sum()
dia["mes_sen"] = np.sin(2 * np.pi * dia.index.month / 12)
dia["mes_cos"] = np.cos(2 * np.pi * dia.index.month / 12)
print(dia.shape, dia.index.min().date(), dia.index.max().date())
print("dias com dado:", int(dia["dia_com_dado"].sum()))

## 2. Rótulo: alagamentos do CGE por dia

In [ ]:
rot = pd.read_csv(ARQ_ROTULO, parse_dates=["data"]).set_index("data")
rot = rot[rot["pagina_ok"].astype(str).str.lower() == "true"]   # página com erro = dado faltante
y_bin, y_cont = f"alagou_{REGIAO}", f"n_pontos_{REGIAO}"

VARS = ["chuva_24h", "chuva_max_1h", "chuva_max_3h", "chuva_dia_anterior", "chuva_72h",
        "chuva_7d_anteriores", "mes_sen", "mes_cos"]
base = dia[dia["dia_com_dado"]].join(rot[[y_bin, y_cont]], how="inner").dropna(subset=VARS + [y_bin])
print("dias na base:", len(base), "| dias com alagamento:", int(base[y_bin].sum()),
      f"({base[y_bin].mean():.1%})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
base.groupby(base.index.year)[y_bin].mean().plot.bar(ax=ax[0], title="Fração de dias com alagamento por ano")
ax[1].boxplot([base.loc[base[y_bin] == 0, "chuva_24h"], base.loc[base[y_bin] == 1, "chuva_24h"]])
ax[1].set_xticks([1, 2], ["sem alagamento", "com alagamento"]); ax[1].set_ylabel("chuva no dia (mm)")
ax[1].set_title("Chuva no dia × alagamento"); plt.tight_layout(); plt.show()

## 3. Divisão por tempo (nunca aleatória)

- **Treino:** 2016–2022.
- **Validação:** 2023, usada só para escolher o limite de probabilidade.
- **Teste:** 2024–2025, avaliado uma única vez no fim.

O modelo nunca vê dias posteriores aos que usa para aprender.

In [ ]:
tr = base.loc[:TREINO_ATE]; va = base.loc[VALIDACAO[0]:VALIDACAO[1]]; te = base.loc["2024-01-01":]
for nome, p in [("treino", tr), ("validação", va), ("teste", te)]:
    print(f"{nome:10s} {len(p):5d} dias, {int(p[y_bin].sum()):3d} com alagamento")

## 4. Regressão logística: vai alagar hoje?

A **referência simples** é a regra "alaga se chover ≥ N mm no dia", com N escolhido no treino pelo melhor F1. O modelo só vale a pena se superar essa regra.

In [ ]:
def metricas(y, pred, prob=None):
    m = {"precisão": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred),
         "F1": f1_score(y, pred)}
    if prob is not None:
        m["PR-AUC"] = average_precision_score(y, prob)
    return m

limiares_mm = np.arange(5, 120, 1)
f1s = [f1_score(tr[y_bin], tr["chuva_24h"] >= n) for n in limiares_mm]
N_REF = float(limiares_mm[int(np.argmax(f1s))])
print(f"Referência: alaga se chuva_24h >= {N_REF:.0f} mm")

logit = Pipeline([("escala", StandardScaler()),
                  ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
logit.fit(tr[VARS], tr[y_bin])

# Limite: na validação, o de maior precisão entre os que têm recall >= RECALL_MINIMO
p_va = logit.predict_proba(va[VARS])[:, 1]
prec, rec, lim = precision_recall_curve(va[y_bin], p_va)
ok = rec[:-1] >= RECALL_MINIMO
LIMITE = float(lim[ok][np.argmax(prec[:-1][ok])]) if ok.any() else 0.5
print(f"Limite de alta probabilidade escolhido na validação: {LIMITE:.3f}")

In [ ]:
p_te = logit.predict_proba(te[VARS])[:, 1]
pred_te = (p_te >= LIMITE).astype(int)
ref_te = (te["chuva_24h"] >= N_REF).astype(int)
tabela = pd.DataFrame({"referência (≥ N mm)": metricas(te[y_bin], ref_te),
                       "regressão logística": metricas(te[y_bin], pred_te, p_te)}).round(3)
display(tabela)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ConfusionMatrixDisplay(confusion_matrix(te[y_bin], ref_te), display_labels=["não", "sim"]).plot(ax=ax[0], colorbar=False)
ax[0].set_title("Referência");
ConfusionMatrixDisplay(confusion_matrix(te[y_bin], pred_te), display_labels=["não", "sim"]).plot(ax=ax[1], colorbar=False)
ax[1].set_title("Logística"); plt.tight_layout(); plt.show()
coef = pd.Series(logit.named_steps["modelo"].coef_[0], index=VARS).sort_values()
print("Coeficientes (variáveis padronizadas):"); print(coef.round(3))

**Interpretação (preencher com os números finais):**
- Quantos dias de alagamento do teste o modelo avisou (recall) e quantos alarmes foram falsos (precisão).
- Comparação com a referência.
- Quais variáveis mais pesam. Espera-se pico em 1 h ou 3 h e chuva acumulada.

## 5. Regressão linear: quantos pontos de alagamento?

A **referência** é a média de pontos por dia no treino.

In [ ]:
lin = Pipeline([("escala", StandardScaler()), ("modelo", LinearRegression())])
lin.fit(tr[VARS], tr[y_cont])
prev = np.clip(lin.predict(te[VARS]), 0, None)          # contagem não pode ser negativa
ref = np.full(len(te), tr[y_cont].mean())
def m_reg(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** 0.5, "R²": r2_score(y, p)}
display(pd.DataFrame({"referência (média)": m_reg(te[y_cont], ref),
                      "regressão linear": m_reg(te[y_cont], prev)}).round(3))
print(pd.Series(lin.named_steps["modelo"].coef_, index=VARS).round(3).sort_values())
plt.figure(figsize=(5, 4)); plt.scatter(te[y_cont], prev, alpha=.4)
plt.xlabel("pontos observados"); plt.ylabel("pontos previstos"); plt.title("Teste 2024–2025"); plt.show()

**Interpretação (preencher):**
- MAE em pontos por dia.
- R² e o que ele diz.
- Coeficientes em mm padronizados.

**Ressalva:** a contagem tem muitos zeros e alta variância, o que não combina bem com uma reta. Isso entra nas limitações.

## 6. Validação no Alto Tietê (Reindesc)

O Alto Tietê não tem rótulo diário. O modelo é aplicado à chuva de uma estação do Alto Tietê e comparado com os dias em que o Cemaden registrou ocorrência hidrológica no município (Reindesc). Rode esta seção quando os CSVs de Mogi e Itaquaquecetuba estiverem em `dados/ped/`.

In [ ]:
# Preencher quando os CSVs chegarem: repetir a seção 1 para a estação, aplicar logit.predict_proba e
# comparar a probabilidade média nos dias com ocorrência no Reindesc × demais dias.

## 7. Exportar o modelo para o serviço Python (`/risk`)

In [ ]:
saida = Path("modelos"); saida.mkdir(exist_ok=True)
joblib.dump({"pipeline": logit, "variaveis": VARS, "limite": LIMITE, "regiao": REGIAO,
             "treino_ate": TREINO_ATE}, saida / "risco_logistica.joblib")
print("salvo em", saida / "risco_logistica.joblib", "| limite =", round(LIMITE, 3))

## 8. Recomendações, limitações e impactos

*(rascunho; ajustar com os números finais)*

### Recomendações
1. **Usar a probabilidade como alerta antecipado, não como substituto do sensor.** O SMS de "alta chance de alagamento" sai antes da água subir. O alerta por nível d'água em cm continua sendo o principal.
2. **Priorizar recall:** o limite foi escolhido para avisar ao menos 70% dos dias de alagamento, aceitando alarmes falsos. Para quem não consegue sair sozinho (idosos, acamados), um aviso a mais custa menos que um aviso a menos.
3. **Revisar o limite a cada estação chuvosa**, com os dados novos de chuva e de alagamento.
4. **Ampliar o rótulo:** acrescentar as ocorrências da Defesa Civil (GeoSampa), que vêm de chamados de moradores e cobrem ruas residenciais, e mais estações do Cemaden.

### Limitações
- **O rótulo mede alagamento em via monitorada.** O CGE registra sobretudo vias principais, informadas por agentes de trânsito, e subnotifica ruas residenciais e a várzea do Tietê (Jardim Helena, Jardim Pantanal).
- **Uma estação representa a região inteira.** Chuva de verão é muito localizada.
- **Granularidade diária:** o modelo diz "hoje deve alagar", não o horário.
- **Lacunas na chuva:** dias com menos de 20 horas de leitura ficam fora. Fuso UTC e unidade em mm foram assumidos a partir da fonte.
- **O registro muda com o tempo:** o esforço de registro e as obras de drenagem mudam ao longo dos anos.
- **Alto Tietê sem rótulo diário:** o modelo é transferido da capital e só checado contra as ocorrências do Reindesc.
- **Regressão linear sobre contagem com muitos zeros:** um modelo de Poisson seria mais adequado (trabalho futuro).

### Impactos
- **Tempo de reação:** o aviso antes da água subir dá tempo para tirar de casa quem precisa de ajuda.
- **Custo do erro:** falso negativo significa alguém sem aviso. Falso positivo significa desgaste e perda de confiança no alerta, por isso a revisão periódica do limite.
- **Ética e dados:** só dados públicos e simulados, sem dados pessoais reais.